# 00 — V3 preflight and v2 instrument re-verification

V3 keeps the completed v2 evidence immutable and asks one new question: can a
smaller, carrying-position intervention retain 3/3 swap efficacy without the
alpha-2 collateral damage? This notebook first records the mandatory fresh
environment checks, then re-runs the bounded v2 sentinels. It does not select
alpha and cannot license science.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

from src.v2_stage0 import collect_preflight, print_preflight

preflight = collect_preflight()
print_preflight(preflight)
assert preflight['status'] == 'PASS', preflight

PATH=/home/jovyan/.local/bin:/home/jovyan/.npm-global/bin:/command:/opt/rsync/usr/bin:/opt/ssh/usr/bin:/opt/ssh/usr/sbin:/opt/sudo/usr/bin:/opt/sudo/usr/sbin:/opt/conda/bin:/opt/conda/condabin:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
codex: /home/jovyan/.npm-global/bin/codex
hf: /home/jovyan/.local/bin/hf
git: /usr/bin/git
nvidia-smi: /usr/bin/nvidia-smi

[codex_version] returncode=0
codex-cli 0.142.5

[hf_auth] returncode=0
user=sushmanth orgs=devoworm-group,sushmanthreddy,OWG,syscv-community,context-course

[git_global_config] returncode=0
user.name=sushmanthreddy
user.email=sushmanthreddymereddy@cisco.com
init.defaultbranch=main
credential.helper=store --file /home/jovyan/.git-credentials

[git_remote] returncode=0
origin	https://github.com/sushmanthreddy/j-space-thoughts.git (fetch)
origin	https://github.com/sushmanthreddy/j-space-thoughts.git (push)

[gpu] returncode=0
name, memory.total [MiB], memory.free [MiB]
NVIDIA H200, 143771 MiB, 143072 Mi

In [2]:
prior = json.loads((ROOT / 'results/metrics.json').read_text())
repair_v2 = prior['repair_v2']
assert repair_v2['gate_ledger']['g_swap'] == 'PASS'
assert repair_v2['gate_ledger']['g_dir'] == 'PASS'
workspace_layers = repair_v2['stage1']['g_swap']['canonical_configuration']['layers']

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)
print({
    'model': bundle.model_id,
    'revision': bundle.revision,
    'dtype': str(next(bundle.hf_model.parameters()).dtype),
    'workspace_layers': workspace_layers,
})

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{'model': 'Qwen/Qwen2.5-7B-Instruct', 'revision': 'a09a35458c702b33eeacc393d103063234e8bc28', 'dtype': 'torch.bfloat16', 'workspace_layers': [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]}


In [3]:
from src.v3_reverify import run_stage0_reverify

stage0 = run_stage0_reverify(
    bundle,
    lens,
    v2_metrics=repair_v2,
    workspace_layers=workspace_layers,
    preflight=preflight,
)
print(json.dumps({
    'status': stage0['status'],
    'checks': stage0['checks'],
    'g1_max_mean_kl': stage0['g1']['max_prompt_mean_kl'],
    'known_swaps': stage0['known_swaps']['n_pass'],
    'gdir_top1': stage0['gdir']['heldout_retrieval_top1'],
    'gdir_top5': stage0['gdir']['known_answer_top5'],
    'controls_fire': stage0['controls_fire']['status'],
}, indent=2))
for row in stage0['known_swaps']['rows']:
    print(row['name'], row['clean_top'], '->', row['edited_top'], row['pass'])

{
  "status": "PASS",
  "checks": {
    "preflight": true,
    "hf_jlens_logits": true,
    "known_swaps": true,
    "gdir_artifact": true,
    "controls_fire": true
  },
  "g1_max_mean_kl": 1.6602172081547906e-08,
  "known_swaps": 3,
  "gdir_top1": 0.55,
  "gdir_top5": 0.8875,
  "controls_fire": "PASS"
}
spider-legs 8 -> 6 True
animal-legs-buffalo2  four ->  eight True
chem-photosynthesis-Z 8 -> 7 True


In [4]:
from src.v3_reverify import persist_stage0

metrics = persist_stage0(stage0)
v3 = metrics['calibration_v3']
assert v3['gate_ledger']['stage0_reverify'] == 'PASS'
assert v3['gate_ledger']['g_swap'] == 'NOT_RUN_V3'
assert v3['gate_ledger']['stage3_science'] == 'PROHIBITED'
print(json.dumps(v3['gate_ledger'], indent=2))

{
  "stage0_reverify": "PASS",
  "g_swap": "NOT_RUN_V3",
  "g_alpha": "NOT_RUN_V3",
  "stage2_recalibration": "PROHIBITED",
  "stage3_science": "PROHIBITED",
  "stage4_report": "NOT_RUN_V3"
}


In [5]:
import gc
import torch

del stage0, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print('Notebook 00 complete. Only notebook 01 G-SWAP is now permitted.')

Notebook 00 complete. Only notebook 01 G-SWAP is now permitted.
